In [2]:
pip install neo4j

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
from db import Neo4jConnection
import pandas as pd
db = Neo4jConnection()


In [55]:
df = pd.read_csv('../data/skills.csv')
# View the first 5 rows
print(df.head())

     Id    Skill
0  S001       C#
1  S002     .NET
2  S003  Angular
3  S004   Python
4  S005    Azure


In [56]:
def create_skill(tx, skill):
    # Notice there is absolutely NO comma after $Skill
    query = """
    CREATE (s:Skill { skill_id: $Id, name: $Skill } ) RETURN s
    """
    tx.run(query, parameters=skill)

In [57]:
with db.driver.session() as session:
    for skill_row in df.to_dict('records'):
        session.execute_write(create_skill, skill_row)

In [58]:
candidates_df = pd.read_csv('../data/candidates.csv')
# View the first 5 rows
print(candidates_df.head())

     Id     name
0  C001  Krishna
1  C002    Rahul
2  C003    Priya


In [59]:
def create_candidate(tx, skill):
    # Notice there is absolutely NO comma after $Skill
    query = """
    CREATE (c:Candidate { candidate_id: $Id, name: $name } ) RETURN c
    """
    tx.run(query, parameters=skill)

In [60]:
with db.driver.session() as session:
    for candi_row in candidates_df.to_dict('records'):
        print(candi_row)
        session.execute_write(create_candidate, candi_row)

{'Id': 'C001', 'name': 'Krishna'}
{'Id': 'C002', 'name': 'Rahul'}
{'Id': 'C003', 'name': 'Priya'}


In [61]:
query = """
MATCH (c:Candidate {candidate_id: $c_id})
MATCH (s:Skill {name: $s_name})
MERGE (c)-[:HAS_SKILL]->(s)
"""

with db.driver.session() as session:
    session.run(
        query,
        c_id="C002",      # Maps to $c_id -> looks up candidate_id: "C002"
        s_name="Azure"    # Maps to $s_name -> looks up name: "Azure"
    )


with db.driver.session() as session:
    session.run(
        query,
        c_id="C001",      # Maps to $c_id -> looks up candidate_id: "C002"
        s_name="Machine Learning"    # Maps to $s_name -> looks up name: "Azure"
    )
    session.run(
            query,
            c_id="C003",      # Maps to $c_id -> looks up candidate_id: "C002"
            s_name="Machine Learning"    # Maps to $s_name -> looks up name: "Azure"
        )

In [62]:
with db.driver.session() as session:
    session.run(
        query,
        c_id="C001",      # Maps to $c_id -> looks up candidate_id: "C002"
        s_name="C#"    # Maps to $s_name -> looks up name: "Azure"
    )
    session.run(
            query,
            c_id="C001",      # Maps to $c_id -> looks up candidate_id: "C002"
            s_name=".NET"    # Maps to $s_name -> looks up name: "Azure"
        )
    session.run(
            query,
            c_id="C001",      # Maps to $c_id -> looks up candidate_id: "C002"
            s_name="Angular"    # Maps to $s_name -> looks up name: "Azure"
        )
    session.run(
                query,
                c_id="C001",      # Maps to $c_id -> looks up candidate_id: "C002"
                s_name="SQL"    # Maps to $s_name -> looks up name: "Azure"
            )


In [64]:
jobs_df = pd.read_csv('../data/jobs.csv')
# View the first 5 rows
print(jobs_df.head())

     Id                   name
0  J001  Senior .NET Developer
1  J002        Azure Developer
2  J003   Full Stack Developer
3  J004            AI Engineer
4  J005          Data Engineer


In [65]:
def create_job(tx, jb):
    # Notice there is absolutely NO comma after $Skill
    query = """
    CREATE (j:Job { job_id: $Id, name: $name } ) RETURN j
    """
    tx.run(query, parameters=jb)

In [66]:
with db.driver.session() as session:
    for job_row in jobs_df.to_dict('records'):
        print(job_row)
        session.execute_write(create_job, job_row)

{'Id': 'J001', 'name': 'Senior .NET Developer'}
{'Id': 'J002', 'name': 'Azure Developer'}
{'Id': 'J003', 'name': 'Full Stack Developer'}
{'Id': 'J004', 'name': 'AI Engineer'}
{'Id': 'J005', 'name': 'Data Engineer'}


In [67]:
# 1. Clean, fixed query string
query = """
MATCH (j:Job {job_id: $j_id})
MATCH (s:Skill {name: $s_name})
MERGE (j)-[:REQUIRES]->(s)
"""

with db.driver.session() as session:
    # 2. Parameters MUST match $j_id and $s_name exactly
    session.run(
        query,
        j_id="J001",      # Changed from c_id to j_id
        s_name="C#"       # Matches $s_name
    )
    session.run(
        query,
        j_id="J001",
        s_name=".NET"
    )
    session.run(
        query,
        j_id="J001",
        s_name="Angular"
    )


In [68]:
with db.driver.session() as session:
     session.run(
            query,
            j_id="J001",
            s_name="SQL"
        )
     session.run(
                 query,
                 j_id="J001",
                 s_name="Azure"
             )

In [69]:
industries_df = pd.read_csv('../data/industries.csv')
# View the first 5 rows
print(industries_df.head())

       Id        name
0  IND001     Banking
1  IND002   Insurance
2  IND003  Healthcare
3  IND004      Retail
4  IND005  Technology


In [70]:
def create_industries(tx, ind):
    # Notice there is absolutely NO comma after $Skill
    query = """
    CREATE (i:Industrie { industrie_id: $Id, name: $name } ) RETURN i
    """
    tx.run(query, parameters=ind)

In [71]:
with db.driver.session() as session:
    for industries_row in industries_df.to_dict('records'):
        print(industries_row)
        session.execute_write(create_industries, industries_row)

{'Id': 'IND001', 'name': 'Banking'}
{'Id': 'IND002', 'name': 'Insurance'}
{'Id': 'IND003', 'name': 'Healthcare'}
{'Id': 'IND004', 'name': 'Retail'}
{'Id': 'IND005', 'name': 'Technology'}


In [72]:
query = """
MATCH (j:Job {job_id: $j_id})
MATCH (s:Industrie {name: $s_name})
MERGE (j)-[:BELONGS_TO]->(s)
"""


In [73]:
with db.driver.session() as session:
    # 2. Parameters MUST match $j_id and $s_name exactly
    session.run(
        query,
        j_id="J001",      # Changed from c_id to j_id
        s_name="Banking"       # Matches $s_name
    )
    session.run(
        query,
        j_id="J001",
        s_name="Insurance"
    )
    session.run(
            query,
            j_id="J001",
            s_name="Healthcare"
        )
    session.run(
                query,
                j_id="J001",
                s_name="Retail"
            )
    
    


In [74]:
companies_df = pd.read_csv('../data/companies.csv')
# View the first 5 rows
print(companies_df.head())

        Id       name
0  COMP001  Microsoft
1  COMP002  Accenture
2  COMP003  Capgemini
3  COMP004   Deloitte


In [75]:
def create_companies(tx, com):
    # Notice there is absolutely NO comma after $Skill
    query = """
    CREATE (c:Companies { companies_id: $Id, name: $name } ) RETURN c
    """
    tx.run(query, parameters=com)

In [76]:
with db.driver.session() as session:
    for companies_row in companies_df.to_dict('records'):
        print(companies_row)
        session.execute_write(create_companies, companies_row)

{'Id': 'COMP001', 'name': 'Microsoft'}
{'Id': 'COMP002', 'name': 'Accenture'}
{'Id': 'COMP003', 'name': 'Capgemini'}
{'Id': 'COMP004', 'name': 'Deloitte'}


In [78]:
query = """
MATCH (j:Job {job_id: $j_id})
MATCH (c:Companies {name: $s_name})
MERGE (j)-[:POSTED_BY]->(c)
"""


In [79]:
with db.driver.session() as session:
    # 2. Parameters MUST match $j_id and $s_name exactly
    session.run(
        query,
        j_id="J001",      # Changed from c_id to j_id
        s_name="Microsoft"       # Matches $s_name
    )
    session.run(
            query,
            j_id="J001",      # Changed from c_id to j_id
            s_name="Capgemini"       # Matches $s_name
        )
    session.run(
                query,
                j_id="J001",      # Changed from c_id to j_id
                s_name="Accenture"       # Matches $s_name
            )
    


In [80]:
location_df = pd.read_csv('../data/locations.csv')
# View the first 5 rows
print(location_df.head())

       Id       name
0  LOC001  Hyderabad
1  LOC002  Bangalore
2  LOC003    Chennai
3  LOC004       Pune
4  LOC005      Delhi


In [81]:
def create_locations(tx, com):
    # Notice there is absolutely NO comma after $Skill
    query = """
    CREATE (c:Locations { location_id: $Id, name: $name } ) RETURN c
    """
    tx.run(query, parameters=com)

In [82]:
with db.driver.session() as session:
    for location_row in location_df.to_dict('records'):
        print(location_row)
        session.execute_write(create_locations, location_row)

{'Id': 'LOC001', 'name': 'Hyderabad'}


[#E158]  _: <CONNECTION> error: Failed to write data to connection IPv4Address(('p-mt-2e58c6e6c448-27-0134.production-orch-0703.neo4j.io', 7687)) (ResolvedIPv4Address(('34.124.169.171', 7687))): SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2406)')
Transaction failed and will be retried in 0.9634809497190714s (Failed to write data to connection IPv4Address(('p-mt-2e58c6e6c448-27-0134.production-orch-0703.neo4j.io', 7687)) (ResolvedIPv4Address(('34.124.169.171', 7687))))


{'Id': 'LOC002', 'name': 'Bangalore'}
{'Id': 'LOC003', 'name': 'Chennai'}
{'Id': 'LOC004', 'name': 'Pune'}
{'Id': 'LOC005', 'name': 'Delhi'}


In [90]:
query = """
MATCH (l:Locations {name: $loc_name})
MATCH (c:Companies {name: $com_name})
MERGE (c)-[:LOCATED_IN]->(l)
"""

with db.driver.session() as session:
    session.run(
        query,
        loc_name="Hyderabad",      
        com_name="Microsoft"       
    )
    session.run(
        query,
        loc_name="Hyderabad",      
        com_name="Capgemini"       
    )


In [93]:
### Select Operations ####
query = """
MATCH (c:Candidate {name: $c_name})-[:HAS_SKILL]->(s:Skill)<-[:REQUIRES]-(j:Job)
RETURN j.name AS job_title, count(s) AS match_score
ORDER BY match_score DESC
"""

with db.driver.session() as session:
    result = session.run(query, c_name="Rahul")
    for record in result:
        print(f"Job: {record['job_title']} | Skills Matched: {record['match_score']}")
with db.driver.session() as session:
    result = session.run(query, c_name="Krishna")
    for record in result:
        print(f"Job: {record['job_title']} | Skills Matched: {record['match_score']}")


Job: Senior .NET Developer | Skills Matched: 1
Job: Senior .NET Developer | Skills Matched: 4


In [94]:
## Location-Based Skill Demand ##
query = """
MATCH (l:Location {name: $location_name})<-[:LOCATED_IN]-(c:Company)
MATCH (c)<-[:POSTED_BY]-(j:Job)-[:REQUIRES]->(s:Skill)
RETURN s.name AS skill_name, count(j) AS job_count
ORDER BY job_count DESC
LIMIT 5
"""

with db.driver.session() as session:
    result = session.run(query, location_name="Hyderabad")
    print("--- In-Demand Skills in Hyderabad ---")
    for record in result:
        print(f"Skill: {record['skill_name']} | Openings: {record['job_count']}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Location` does not exist in database `93dd8a27`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=10, offset=10>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 2, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (l:Location {name: $location_name})<-[:LOCATED_IN]-(c:Company)\nMATCH (c)<-[:POSTED_BY]-(j:Job)-[:REQUIRES]->(s:Skill)\nRETURN s.name AS skill_name, count(j) AS job_count\nORDER BY job_count DESC\nLIMIT 5\n'


--- In-Demand Skills in Hyderabad ---


In [95]:
## Industry Popularity Contest ##
query = """
MATCH (i:Industry)<-[:BELONGS_TO]-(c:Company)<-[:POSTED_BY]-(j:Job)
RETURN i.name AS industry_name, count(j) AS total_jobs
ORDER BY total_jobs DESC
"""

with db.driver.session() as session:
    result = session.run(query)
    print("--- Job Volume by Industry ---")
    for record in result:
        print(f"Industry: {record['industry_name']} | Total Jobs: {record['total_jobs']}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Industry` does not exist in database `93dd8a27`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=10, offset=10>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 2, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (i:Industry)<-[:BELONGS_TO]-(c:Company)<-[:POSTED_BY]-(j:Job)\nRETURN i.name AS industry_name, count(j) AS total_jobs\nORDER BY total_jobs DESC\n'


--- Job Volume by Industry ---


In [96]:
# Co-occurring Skill Recommendations (Skill-to-Skill)
query = """
MATCH (s1:Skill {name: $target_skill})<-[:REQUIRES]-(j:Job)-[:REQUIRES]->(s2:Skill)
WHERE s1 <> s2
RETURN s2.name AS related_skill, count(j) AS frequency
ORDER BY frequency DESC
LIMIT 3
"""

with db.driver.session() as session:
    result = session.run(query, target_skill="Azure")
    print("--- Skills frequently paired with Azure ---")
    for record in result:
        print(f"Related Skill: {record['related_skill']} (Appears in {record['frequency']} jobs)")


--- Skills frequently paired with Azure ---
Related Skill: .NET (Appears in 1 jobs)
Related Skill: Angular (Appears in 1 jobs)
Related Skill: C# (Appears in 1 jobs)
